# 🔬 MedVQA Niveau 5 (N5) — ViT-Base/16 + BiomedBERT + Concat Simple
### Première intégration multimodale ViT + BiomedBERT avec optimisations mémoire anti-OOM
---
- **Backbone Visuel** : ViT-Base/16 (224x224)
- **Backbone Textuel** : BiomedBERT
- **Fusion** : Concaténation simple + MLP
- **Dataset** : VQA-RAD + SLAKE (~7 000 échantillons)


In [ ]:
# -*- coding: utf-8 -*-
"""MedVQA_N5_Colab_Fixed_v2.py

N5 Minimal — ViT-B/16 + BiomedBERT + concat + MLP
Optimisé pour Google Colab (GPU T4)

FIXES v2 :
  - Chargement séquentiel : VQA-RAD chargé + libéré AVANT SLAKE
  - Images stockées en bytes JPEG (~50x moins de RAM que PIL)
  - PIL reconstruit à la volée dans __getitem__ puis libéré
  - GradScaler / autocast conditionnels (CPU-safe)
  - gc.collect() + cuda.empty_cache() aux moments critiques
  - pin_memory / non_blocking conditionnels
  - Affichage VRAM à chaque epoch
"""


### 📌 INSTALLATION

In [ ]:
# =============================================================================
# INSTALLATION
# =============================================================================
!pip install transformers datasets torchvision timm sacrebleu scikit-learn -q

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler

import torchvision.transforms as transforms
import timm
from transformers import AutoModel, AutoTokenizer
from datasets import load_dataset

import numpy as np
from PIL import Image
import io
import gc
import random
from collections import Counter


### 📌 DEVICE + VÉRIFICATION GPU

In [ ]:
# =============================================================================
# DEVICE + VÉRIFICATION GPU
# =============================================================================
device  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = device.type == 'cuda'
PIN_MEM = device.type == 'cuda'

print('=' * 55)
print(f'  Device  : {device}')
if device.type == 'cuda':
    print(f'  GPU     : {torch.cuda.get_device_name(0)}')
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'  VRAM    : {vram:.1f} GB')
    print(f'  AMP     : activé')
else:
    print('  ⚠️  Pas de GPU — Exécution > Modifier le type d\'exécution > GPU T4')
print('=' * 55)

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)


### 📌 UTILITAIRE : PIL → bytes JPEG compressés (~50x moins de RAM)

In [ ]:
# =============================================================================
# UTILITAIRE : PIL → bytes JPEG compressés (~50x moins de RAM)
# =============================================================================
def pil_to_bytes(pil_img):
    buf = io.BytesIO()
    pil_img.convert('RGB').save(buf, format='JPEG', quality=85)
    return buf.getvalue()


### 📌 NORMALISEURS

In [ ]:
# =============================================================================
# NORMALISEURS
# =============================================================================
def normalize_vqa_rad(sample):
    answer = str(sample['answer']).strip().lower()
    answer_type = 'CLOSED' if answer in ('yes', 'no') else 'OPEN'
    return {
        'image_bytes': pil_to_bytes(sample['image']),   # bytes, pas PIL
        'question'   : str(sample['question']).strip(),
        'answer'     : answer,
        'answer_type': answer_type,
        'question_type': 'OTHER',
    }

def normalize_slake(sample):
    answer = str(sample['answer']).strip().lower()
    raw_type = sample.get('answer_type', '')
    answer_type = 'CLOSED' if str(raw_type).strip().lower() == 'closed' else 'OPEN'
    return {
        'image_bytes': pil_to_bytes(sample['image']),   # bytes, pas PIL
        'question'   : str(sample['question']).strip(),
        'answer'     : answer,
        'answer_type': answer_type,
        'question_type': str(sample.get('qid', 'OTHER')),
    }


### 📌 ÉTAPE 1 : Chargement SÉQUENTIEL (anti-OOM)

In [ ]:
# =============================================================================
# ÉTAPE 1 : Chargement SÉQUENTIEL (anti-OOM)
#   → charger un dataset, convertir, supprimer le brut, gc.collect()
#   → puis passer au suivant
# =============================================================================
print("\n=== Chargement VQA-RAD ===")
_vqa = load_dataset("flaviagiammarino/vqa-rad")
vqa_rad_train = [normalize_vqa_rad(s) for s in _vqa['train']]
vqa_rad_test  = [normalize_vqa_rad(s) for s in _vqa['test']]
del _vqa
gc.collect()
print(f"VQA-RAD — Train: {len(vqa_rad_train)} | Test: {len(vqa_rad_test)}")
print(f"RAM images VQA-RAD ≈ {sum(len(s['image_bytes']) for s in vqa_rad_train) / 1e6:.1f} MB")

print("\n=== Chargement SLAKE ===")
_slake = load_dataset("mdwiratathya/SLAKE-vqa-english")
slake_train = [normalize_slake(s) for s in _slake['train']]
slake_val   = [normalize_slake(s) for s in _slake['validation']]
slake_test  = [normalize_slake(s) for s in _slake['test']]
del _slake
gc.collect()
print(f"SLAKE — Train: {len(slake_train)} | Val: {len(slake_val)} | Test: {len(slake_test)}")
print(f"RAM images SLAKE ≈ {sum(len(s['image_bytes']) for s in slake_train) / 1e6:.1f} MB")


### 📌 ÉTAPE 2 : Combiner

In [ ]:
# =============================================================================
# ÉTAPE 2 : Combiner
# =============================================================================
combined_train = vqa_rad_train + slake_train
combined_val   = slake_val
combined_test  = vqa_rad_test + slake_test

# Libérer les références intermédiaires
del vqa_rad_train, slake_train, slake_val, vqa_rad_test, slake_test
gc.collect()

print(f"\nCOMBINÉ — Train: {len(combined_train)} | Val: {len(combined_val)} | Test: {len(combined_test)}")


### 📌 ÉTAPE 3 : Vocabulaires

In [ ]:
# =============================================================================
# ÉTAPE 3 : Vocabulaires
# =============================================================================
closed_counter = Counter(s['answer'] for s in combined_train if s['answer_type'] == 'CLOSED')
open_counter   = Counter(s['answer'] for s in combined_train if s['answer_type'] == 'OPEN')

TOP_CLOSED = 500
TOP_OPEN   = 1000

top_closed = [ans for ans, _ in closed_counter.most_common(TOP_CLOSED)]
top_open   = [ans for ans, _ in open_counter.most_common(TOP_OPEN)]

closed2idx = {a: i for i, a in enumerate(top_closed)}
open2idx   = {a: i for i, a in enumerate(top_open)}
idx2closed = {i: a for a, i in closed2idx.items()}
idx2open   = {i: a for a, i in open2idx.items()}

NUM_CLOSED = len(closed2idx)
NUM_OPEN   = len(open2idx)

print(f"\nVocabulaires — Closed: {NUM_CLOSED} | Open: {NUM_OPEN}")

def in_vocab(s):
    if s['answer_type'] == 'CLOSED':
        return s['answer'] in closed2idx
    return s['answer'] in open2idx

combined_train = [s for s in combined_train if in_vocab(s)]
combined_val   = [s for s in combined_val   if in_vocab(s)]
combined_test  = [s for s in combined_test  if in_vocab(s)]

print(f"Après filtrage — Train: {len(combined_train)} | Val: {len(combined_val)} | Test: {len(combined_test)}")


### 📌 ÉTAPE 4 : Dataset & Transforms

In [ ]:
# =============================================================================
# ÉTAPE 4 : Dataset & Transforms
# =============================================================================
MEAN = [0.5, 0.5, 0.5]
STD  = [0.5, 0.5, 0.5]

train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD),
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD),
])

class MedVQADataset(Dataset):
    def __init__(self, data, transform, closed2idx, open2idx):
        self.data = data
        self.transform = transform
        self.c2i = closed2idx
        self.o2i = open2idx

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        s = self.data[idx]
        # Reconstruire PIL depuis bytes → transformer → PIL libérée automatiquement
        image = Image.open(io.BytesIO(s['image_bytes'])).convert('RGB')
        image = self.transform(image)
        question  = s['question']
        is_closed = (s['answer_type'] == 'CLOSED')
        answer_type = 0 if is_closed else 1
        label = self.c2i[s['answer']] if is_closed else self.o2i[s['answer']]
        qtype = s.get('question_type', 'OTHER')
        return image, question, answer_type, label, qtype


### 📌 ÉTAPE 5 : Tokenizer — BiomedBERT

In [ ]:
# =============================================================================
# ÉTAPE 5 : Tokenizer — BiomedBERT
# =============================================================================
BIOMEDBERT = "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract"
tokenizer  = AutoTokenizer.from_pretrained(BIOMEDBERT)

def collate_fn(batch):
    images, questions, ans_types, labels, qtypes = zip(*batch)
    images    = torch.stack(images)
    tokens    = tokenizer(list(questions), padding=True, truncation=True,
                          max_length=64, return_tensors='pt')
    ans_types = torch.tensor(ans_types, dtype=torch.long)
    labels    = torch.tensor(labels,    dtype=torch.long)
    return images, tokens['input_ids'], tokens['attention_mask'], ans_types, labels, list(qtypes)

train_dataset = MedVQADataset(combined_train, train_transform, closed2idx, open2idx)
val_dataset   = MedVQADataset(combined_val,   val_transform,   closed2idx, open2idx)
test_dataset  = MedVQADataset(combined_test,  val_transform,   closed2idx, open2idx)

BATCH_SIZE = 16

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          collate_fn=collate_fn, num_workers=2, pin_memory=PIN_MEM)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          collate_fn=collate_fn, num_workers=2, pin_memory=PIN_MEM)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False,
                          collate_fn=collate_fn, num_workers=2, pin_memory=PIN_MEM)

print(f"\nLoaders — Train: {len(train_loader)} | Val: {len(val_loader)} | Test: {len(test_loader)}")
print(f"batch_size={BATCH_SIZE} | AMP={USE_AMP} | pin_memory={PIN_MEM}")


### 📌 ÉTAPE 6 : Modèle N5 Minimal

In [ ]:
# =============================================================================
# ÉTAPE 6 : Modèle N5 Minimal
# =============================================================================
class MedVQA_N5_Minimal(nn.Module):
    def __init__(self, num_closed, num_open, dim=768):
        super().__init__()
        self.visual_encoder = timm.create_model(
            'vit_base_patch16_224', pretrained=True, num_classes=0
        )
        self.text_encoder = AutoModel.from_pretrained(BIOMEDBERT)
        self.fusion = nn.Sequential(
            nn.Linear(dim * 2, dim),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(dim, dim // 2),
            nn.ReLU(),
            nn.Dropout(0.3),
        )
        self.head_closed = nn.Linear(dim // 2, num_closed)
        self.head_open   = nn.Sequential(
            nn.Linear(dim // 2, dim // 4),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(dim // 4, num_open),
        )

    def forward(self, img, input_ids, attn_mask):
        v = self.visual_encoder.forward_features(img)
        v = v.mean(dim=1)
        q = self.text_encoder(input_ids=input_ids, attention_mask=attn_mask)
        q = q.last_hidden_state[:, 0, :]
        fused = torch.cat([v, q], dim=-1)
        fused = self.fusion(fused)
        return self.head_closed(fused), self.head_open(fused)

gc.collect()
if device.type == 'cuda':
    torch.cuda.empty_cache()

model = MedVQA_N5_Minimal(num_closed=NUM_CLOSED, num_open=NUM_OPEN).to(device)
print(f"\nParams totaux : {sum(p.numel() for p in model.parameters()):,}")

if device.type == 'cuda':
    used  = torch.cuda.memory_allocated() / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM après chargement modèle : {used:.1f} / {total:.1f} GB")


### 📌 ÉTAPE 7 : Entraînement

In [ ]:
# =============================================================================
# ÉTAPE 7 : Entraînement
# =============================================================================
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# Phase 1 : encodeurs gelés
for p in model.visual_encoder.parameters():
    p.requires_grad = False
for p in model.text_encoder.parameters():
    p.requires_grad = False

optimizer = torch.optim.AdamW([
    {'params': model.fusion.parameters(),      'lr': 1e-3},
    {'params': model.head_closed.parameters(), 'lr': 1e-3},
    {'params': model.head_open.parameters(),   'lr': 1e-3},
], weight_decay=1e-2)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=3)

scaler = GradScaler('cuda') if USE_AMP else None

best_val_closed  = 0.0
patience_limit   = 10
epochs_no_improve = 0
EPOCHS = 40

print("\n=== Phase 1 : Encodeurs GELÉS ===")

for epoch in range(EPOCHS):

    # Phase 2 : dégel à epoch 5
    if epoch == 5:
        print("\n>>> Phase 2 : DÉGEL encodeurs <<<")
        for p in model.visual_encoder.parameters():
            p.requires_grad = True
        for p in model.text_encoder.parameters():
            p.requires_grad = True
        optimizer.add_param_group({'params': model.visual_encoder.parameters(), 'lr': 1e-5})
        optimizer.add_param_group({'params': model.text_encoder.parameters(),   'lr': 1e-5})
        gc.collect()
        if device.type == 'cuda':
            torch.cuda.empty_cache()

    # ── TRAIN ────────────────────────────────────────────────────────────────
    model.train()
    total_loss = 0
    correct_c, total_c = 0, 0
    correct_o, total_o = 0, 0

    for imgs, input_ids, attn_mask, ans_types, labels, _ in train_loader:
        imgs      = imgs.to(device,      non_blocking=PIN_MEM)
        input_ids = input_ids.to(device, non_blocking=PIN_MEM)
        attn_mask = attn_mask.to(device, non_blocking=PIN_MEM)
        ans_types = ans_types.to(device, non_blocking=PIN_MEM)
        labels    = labels.to(device,    non_blocking=PIN_MEM)

        optimizer.zero_grad()
        mask_c = (ans_types == 0)
        mask_o = (ans_types == 1)

        if USE_AMP:
            with autocast('cuda'):
                logits_c, logits_o = model(imgs, input_ids, attn_mask)
                loss = torch.tensor(0.0, device=device)
                if mask_c.any():
                    loss = loss + criterion(logits_c[mask_c], labels[mask_c])
                if mask_o.any():
                    loss = loss + criterion(logits_o[mask_o], labels[mask_o])
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            logits_c, logits_o = model(imgs, input_ids, attn_mask)
            loss = torch.tensor(0.0, device=device)
            if mask_c.any():
                loss = loss + criterion(logits_c[mask_c], labels[mask_c])
            if mask_o.any():
                loss = loss + criterion(logits_o[mask_o], labels[mask_o])
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        total_loss += loss.item()
        with torch.no_grad():
            if mask_c.any():
                correct_c += (logits_c[mask_c].argmax(1) == labels[mask_c]).sum().item()
                total_c   += mask_c.sum().item()
            if mask_o.any():
                correct_o += (logits_o[mask_o].argmax(1) == labels[mask_o]).sum().item()
                total_o   += mask_o.sum().item()

    # ── VALIDATION ───────────────────────────────────────────────────────────
    model.eval()
    val_correct_c, val_total_c = 0, 0
    val_correct_o, val_total_o = 0, 0

    with torch.no_grad():
        for imgs, input_ids, attn_mask, ans_types, labels, _ in val_loader:
            imgs      = imgs.to(device,      non_blocking=PIN_MEM)
            input_ids = input_ids.to(device, non_blocking=PIN_MEM)
            attn_mask = attn_mask.to(device, non_blocking=PIN_MEM)
            ans_types = ans_types.to(device, non_blocking=PIN_MEM)
            labels    = labels.to(device,    non_blocking=PIN_MEM)
            logits_c, logits_o = model(imgs, input_ids, attn_mask)
            mask_c = (ans_types == 0)
            mask_o = (ans_types == 1)
            if mask_c.any():
                val_correct_c += (logits_c[mask_c].argmax(1) == labels[mask_c]).sum().item()
                val_total_c   += mask_c.sum().item()
            if mask_o.any():
                val_correct_o += (logits_o[mask_o].argmax(1) == labels[mask_o]).sum().item()
                val_total_o   += mask_o.sum().item()

    # ── MÉTRIQUES ────────────────────────────────────────────────────────────
    train_acc_c = 100 * correct_c / total_c         if total_c > 0 else 0
    train_acc_o = 100 * correct_o / total_o         if total_o > 0 else 0
    val_acc_c   = 100 * val_correct_c / val_total_c if val_total_c > 0 else 0
    val_acc_o   = 100 * val_correct_o / val_total_o if val_total_o > 0 else 0

    # Métrique combinée : évite l'early stopping prématuré quand
    # val_acc_c reste à 0% (vocab closed = 2 classes seulement)
    n_c = val_total_c if val_total_c > 0 else 1
    n_o = val_total_o if val_total_o > 0 else 1
    val_acc_combined = 100 * (val_correct_c + val_correct_o) / (n_c + n_o)

    scheduler.step(val_acc_combined)

    if val_acc_combined > best_val_closed:
        best_val_closed = val_acc_combined
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'val_acc_closed':   val_acc_c,
            'val_acc_open':     val_acc_o,
            'val_acc_combined': val_acc_combined,
        }, 'best_n5_minimal.pth')
        saved = " ✓ SAVED"
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
        saved = f" ({epochs_no_improve}/{patience_limit})"

    vram_str = ""
    if device.type == 'cuda':
        vram_str = f" | VRAM {torch.cuda.memory_allocated()/1e9:.1f}GB"

    print(f"Epoch {epoch+1:02d}/{EPOCHS} | Loss: {total_loss/len(train_loader):.4f} | "
          f"Train C:{train_acc_c:.1f}% O:{train_acc_o:.1f}% | "
          f"Val C:{val_acc_c:.1f}% O:{val_acc_o:.1f}% Comb:{val_acc_combined:.1f}%"
          f"{saved}{vram_str}")

    if epochs_no_improve >= patience_limit:
        print(f"\nEarly stopping à l'epoch {epoch+1}")
        break

print(f"\nBest Val Combined: {best_val_closed:.1f}%")


### 📌 ÉTAPE 8 : Test

In [ ]:
# =============================================================================
# ÉTAPE 8 : Test
# =============================================================================
gc.collect()
if device.type == 'cuda':
    torch.cuda.empty_cache()

ck = torch.load('best_n5_minimal.pth', map_location=device)
model.load_state_dict(ck['model_state_dict'])
model.eval()

all_preds_c, all_labels_c = [], []
all_preds_o, all_labels_o = [], []

with torch.no_grad():
    for imgs, input_ids, attn_mask, ans_types, labels, _ in test_loader:
        imgs      = imgs.to(device,      non_blocking=PIN_MEM)
        input_ids = input_ids.to(device, non_blocking=PIN_MEM)
        attn_mask = attn_mask.to(device, non_blocking=PIN_MEM)
        logits_c, logits_o = model(imgs, input_ids, attn_mask)
        mask_c = (ans_types == 0)
        mask_o = (ans_types == 1)
        if mask_c.any():
            all_preds_c.extend(logits_c[mask_c].argmax(1).cpu().tolist())
            all_labels_c.extend(labels[mask_c].tolist())
        if mask_o.any():
            all_preds_o.extend(logits_o[mask_o].argmax(1).cpu().tolist())
            all_labels_o.extend(labels[mask_o].tolist())

closed_acc = (np.array(all_preds_c) == np.array(all_labels_c)).mean() * 100
open_acc   = (np.array(all_preds_o) == np.array(all_labels_o)).mean() * 100

print("\n" + "=" * 60)
print("  TEST RESULTS — N5 Minimal (ViT + BiomedBERT)")
print("=" * 60)
print(f"  Closed Accuracy : {closed_acc:.1f}%  (N3: 57.35%)")
print(f"  Open   Accuracy : {open_acc:.1f}%   (N3: 39.76%)")
print("=" * 60)

if closed_acc > 65:
    print("  ✅ Gain significatif vs N3 — ajouter Cross-Attention + BAN")
elif closed_acc > 60:
    print("  ⚠️  Gain modéré — ajouter BAN pour viser 75%+")
else:
    print("  ❌ Pas de gain — vérifier pré-entraînement ou augmentation")
